<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Application-Deployment/blob/main/CS/04_multipthreading_multiprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### GIL 및 CPU / I-O Bound 속도 비교 (threading vs multiprocessing)
- GIL로 인해 CPU 연산 시 스레드는 속도 이점이 없지만, 프로세스는 멀티코어를 활용해 빨라짐

In [1]:
import time
import threading
import multiprocessing

# =========================================================
# CPU-bound 작업
# =========================================================
def cpu_task(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

# =========================================================
# I/O-bound 작업
# =========================================================
def io_task(seconds):
    time.sleep(seconds)
    return seconds

# =========================================================
# Thread 테스트
# =========================================================
def test_thread_cpu(n, workers=2):
    threads = []
    results = [None] * workers

    def worker(index):
        results[index] = cpu_task(n)

    # 실제 계산 시간만 측정
    start = time.perf_counter()

    for i in range(workers):
        t = threading.Thread(target=worker, args=(i,))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    elapsed = time.perf_counter() - start

    print(f"Thread       : {elapsed:.2f}초")

# =========================================================
# Process 테스트
# =========================================================
def test_process_cpu(n, workers=2):
    # Process 생성 시간을 계산 시간에서 제외
    pool = multiprocessing.Pool(processes=workers)

    # Process가 이미 만들어진 상태에서 측정
    start = time.perf_counter()

    results = pool.map(cpu_task, [n] * workers)

    elapsed = time.perf_counter() - start

    pool.close()
    pool.join()

    print(f"Multiprocess : {elapsed:.2f}초")


# =========================================================
# I/O - Thread
# =========================================================
def test_thread_io(workers=2):

    threads = []

    start = time.perf_counter()

    for _ in range(workers):
        t = threading.Thread(target=io_task, args=(1,))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    elapsed = time.perf_counter() - start

    print(f"Thread       : {elapsed:.2f}초")


# =========================================================
# I/O - Process
# =========================================================
def test_process_io(workers=2):

    pool = multiprocessing.Pool(processes=workers)

    start = time.perf_counter()

    pool.map(io_task, [1] * workers)

    elapsed = time.perf_counter() - start

    pool.close()
    pool.join()

    print(f"Multiprocess : {elapsed:.2f}초")


# =========================================================
# 실행
# =========================================================
if __name__ == "__main__":

    N = 50_000_000
    WORKERS = 2

    print("=" * 50)
    print("CPU-bound")
    print("=" * 50)

    test_thread_cpu(N, WORKERS)
    test_process_cpu(N, WORKERS)

    print()
    print("=" * 50)
    print("I/O-bound")
    print("=" * 50)

    test_thread_io(WORKERS)
    test_process_io(WORKERS)

CPU-bound
Thread       : 13.04초
Multiprocess : 12.93초

I/O-bound
Thread       : 1.00초
Multiprocess : 1.00초
